In [1]:
import requests
import time
import xml.etree.ElementTree as ET
import psycopg2
from psycopg2 import sql

# FatSecret API credentials
client_id = '3af760255a2a4932a779f102f2d80b6b'
client_secret = '933419a37c0943018c52066e1feedc20'
auth_url = 'https://oauth.fatsecret.com/connect/token'
api_url = 'https://platform.fatsecret.com/rest/server.api'  # Updated API URL

# PostgreSQL database credentials
db_config = {
    'dbname': 'recipe_db',
    'user': 'postgres',
    'password': 'anckul7*K',  # Replace with your actual password
    'host': 'localhost',
    'port': '5433'  # Adjust if necessary
}

# Function to get access token
def get_access_token():
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': 'basic'
    }
    response = requests.post(auth_url, data=data)
    
    if response.status_code == 200:
        access_token = response.json().get('access_token')
        print("Access token obtained successfully.")
        return access_token
    else:
        print("Error getting access token:", response.status_code, response.text)
        return None

# Function to save recipe data to the PostgreSQL database
def save_recipe_to_db(recipe_data):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()

        # Insert into recipes table
        cursor.execute(
            """
            INSERT INTO recipes (name, description, servings, prep_time, cook_time)
            VALUES (%s, %s, %s, %s, %s) RETURNING recipe_id;
            """,
            (recipe_data['name'], recipe_data['description'], recipe_data['servings'], recipe_data['prep_time'], recipe_data['cook_time'])
        )
        recipe_id = cursor.fetchone()[0]

        # Insert into nutrition_facts table
        cursor.execute(
            """
            INSERT INTO nutrition_facts (recipe_id, calories, total_fat, protein, carbs)
            VALUES (%s, %s, %s, %s, %s);
            """,
            (recipe_id, recipe_data['calories'], recipe_data['total_fat'], recipe_data['protein'], recipe_data['carbs'])
        )

        # Insert ingredients
        for ingredient in recipe_data['ingredients']:
            cursor.execute(
                """
                INSERT INTO ingredients (recipe_id, ingredient_name, quantity, unit)
                VALUES (%s, %s, %s, %s);
                """,
                (recipe_id, ingredient['name'], ingredient['quantity'], ingredient['unit'])
            )

        # Insert directions
        for step in recipe_data['directions']:
            cursor.execute(
                """
                INSERT INTO directions (recipe_id, instruction_text)
                VALUES (%s, %s);
                """,
                (recipe_id, step)
            )

        # Commit transaction
        conn.commit()
        cursor.close()
        conn.close()
        print(f"Recipe '{recipe_data['name']}' saved to database.")

    except Exception as e:
        print("Error saving recipe to database:", e)

# Function to fetch recipes from FatSecret API
def fetch_recipes():
    access_token = get_access_token()
    
    if not access_token:
        print("Failed to get access token.")
        return
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    
    page_number = 0  
    max_results = 10  
    
    while True:
        params = {
            'method': 'recipes.search',
            'page_number': page_number,
            'max_results': max_results,
            'format': 'xml'
        }
        
        response = requests.get(api_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching page {page_number}: {response.status_code} - {response.text}")
            break

        # Parse XML response with namespace
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            recipes = root.findall('.//ns:recipe', namespace)
            
            if not recipes:
                print(f"No recipes found on page {page_number}.")
                break

            # Process each recipe
            for recipe in recipes:
                recipe_data = {
                    'name': recipe.find('ns:recipe_name', namespace).text,
                    'description': recipe.find('ns:recipe_description', namespace).text,
                    'servings': int(recipe.find('ns:number_of_servings', namespace).text) if recipe.find('ns:number_of_servings', namespace) is not None else None,
                    'prep_time': int(recipe.find('ns:preparation_time_min', namespace).text) if recipe.find('ns:preparation_time_min', namespace) is not None else None,
                    'cook_time': int(recipe.find('ns:cooking_time_min', namespace).text) if recipe.find('ns:cooking_time_min', namespace) is not None else None,
                    'calories': int(recipe.find('.//ns:calories', namespace).text) if recipe.find('.//ns:calories', namespace) is not None else None,
                    'total_fat': float(recipe.find('.//ns:fat', namespace).text) if recipe.find('.//ns:fat', namespace) is not None else None,
                    'protein': float(recipe.find('.//ns:protein', namespace).text) if recipe.find('.//ns:protein', namespace) is not None else None,
                    'carbs': float(recipe.find('.//ns:carbohydrate', namespace).text) if recipe.find('.//ns:carbohydrate', namespace) is not None else None,
                    'ingredients': [
                        {'name': ing.text, 'quantity': float(ing.attrib.get('quantity', 0)), 'unit': ing.attrib.get('unit', '')}
                        for ing in recipe.findall('.//ns:ingredient', namespace)
                    ],
                    'directions': [
                        step.text for step in recipe.findall('.//ns:direction', namespace)
                    ]
                }
                save_recipe_to_db(recipe_data)

            page_number += 1
            print(f"Fetched page {page_number}.")
            time.sleep(2)  # Rate limiting

        except ET.ParseError:
            print("Failed to parse XML response.")
            break

# Start fetching recipes
fetch_recipes()


Access token obtained successfully.
No recipes found on page 0.


In [8]:
import requests
import time
import xml.etree.ElementTree as ET
import psycopg2
from psycopg2 import sql

# FatSecret API credentials
client_id = '3af760255a2a4932a779f102f2d80b6b'
client_secret = '933419a37c0943018c52066e1feedc20'
auth_url = 'https://oauth.fatsecret.com/connect/token'
api_url = 'https://platform.fatsecret.com/rest/server.api'

# PostgreSQL database credentials
db_config = {
    'dbname': 'recipe_db',
    'user': 'postgres',
    'password': '',  # Replace with your actual password
    'host': 'localhost',
    'port': '5433'
}

# Function to get access token
def get_access_token():
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': 'basic'
    }
    response = requests.post(auth_url, data=data)
    
    if response.status_code == 200:
        access_token = response.json().get('access_token')
        print("Access token obtained successfully.")
        return access_token
    else:
        print("Error getting access token:", response.status_code, response.text)
        return None

# Function to check for duplicate recipes by name
def recipe_exists(recipe_name):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Check if recipe exists by name
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_name,))
        exists = cursor.fetchone() is not None

        cursor.close()
        conn.close()
        return exists

    except Exception as e:
        print("Error checking recipe existence:", e)
        return False

# Function to save recipe data to the PostgreSQL database
def save_recipe_to_db(recipe_data):
    try:
        # Prevent duplicate insertions
        if recipe_exists(recipe_data['name']):
            print(f"Recipe '{recipe_data['name']}' already exists. Skipping insert.")
            return

        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()

        # Insert into recipes table
        cursor.execute(
            """
            INSERT INTO recipes (name, description, servings, prep_time, cook_time)
            VALUES (%s, %s, %s, %s, %s) RETURNING recipe_id;
            """,
            (recipe_data['name'], recipe_data['description'], recipe_data['servings'], recipe_data['prep_time'], recipe_data['cook_time'])
        )
        recipe_id = cursor.fetchone()[0]

        # Insert into nutrition_facts table
        cursor.execute(
            """
            INSERT INTO nutrition_facts (recipe_id, calories, total_fat, protein, carbs)
            VALUES (%s, %s, %s, %s, %s);
            """,
            (recipe_id, recipe_data['calories'], recipe_data['total_fat'], recipe_data['protein'], recipe_data['carbs'])
        )

        # Insert ingredients
        for ingredient in recipe_data['ingredients']:
            cursor.execute(
                """
                INSERT INTO ingredients (recipe_id, ingredient_name, quantity, unit)
                VALUES (%s, %s, %s, %s);
                """,
                (recipe_id, ingredient['name'], ingredient['quantity'], ingredient['unit'])
            )

        # Insert directions
        for step in recipe_data['directions']:
            cursor.execute(
                """
                INSERT INTO directions (recipe_id, instruction_text)
                VALUES (%s, %s);
                """,
                (recipe_id, step)
            )

        # Commit transaction
        conn.commit()
        cursor.close()
        conn.close()
        print(f"Recipe '{recipe_data['name']}' saved to database.")

    except Exception as e:
        print("Error saving recipe to database:", e)

# Function to fetch recipes from FatSecret API
def fetch_recipes():
    access_token = get_access_token()
    
    if not access_token:
        print("Failed to get access token.")
        return
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    
    page_number = 0  
    max_results = 10  
    
    while True:
        params = {
            'method': 'recipes.search',
            'page_number': page_number,
            'max_results': max_results,
            'format': 'xml'
        }
        
        response = requests.get(api_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching page {page_number}: {response.status_code} - {response.text}")
            break

        # Parse XML response with namespace
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            recipes = root.findall('.//ns:recipe', namespace)
            
            if not recipes:
                print(f"No recipes found on page {page_number}.")
                break

            # Process each recipe by getting its full details
            for recipe in recipes:
                recipe_id = recipe.find('ns:recipe_id', namespace).text
                fetch_recipe_details(recipe_id, headers)  # Fetch full details for each recipe

            page_number += 1
            print(f"Fetched page {page_number}.")
            time.sleep(2)

        except ET.ParseError:
            print("Failed to parse XML response.")
            break

# New function to fetch full details for each recipe by ID
def fetch_recipe_details(recipe_id, headers):
    params = {
        'method': 'recipe.get',
        'recipe_id': recipe_id,
        'format': 'xml'
    }
    
    response = requests.get(api_url, headers=headers, params=params)
    
    if response.status_code == 200:
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            
            def parse_float(element, namespace, tag):
                text = element.find(tag, namespace).text if element.find(tag, namespace) is not None else None
                return float(text) if text is not None else None
            
            recipe_data = {
                'name': root.find('.//ns:recipe_name', namespace).text if root.find('.//ns:recipe_name', namespace) is not None else "Unnamed Recipe",
                'description': root.find('.//ns:recipe_description', namespace).text if root.find('.//ns:recipe_description', namespace) is not None else "",
                'servings': parse_float(root, namespace, './/ns:number_of_servings'),
                'prep_time': parse_float(root, namespace, './/ns:preparation_time_min'),
                'cook_time': parse_float(root, namespace, './/ns:cooking_time_min'),
                'calories': parse_float(root, namespace, './/ns:calories'),
                'total_fat': parse_float(root, namespace, './/ns:fat'),
                'protein': parse_float(root, namespace, './/ns:protein'),
                'carbs': parse_float(root, namespace, './/ns:carbohydrate'),
                'ingredients': [
                    {
                        'name': ing.find('ns:food_name', namespace).text if ing.find('ns:food_name', namespace) is not None else "Unknown Ingredient",
                        'quantity': float(ing.find('ns:quantity', namespace).text) if ing.find('ns:quantity', namespace) is not None else 0,
                        'unit': ing.find('ns:measure', namespace).text if ing.find('ns:measure', namespace) is not None else ""
                    }
                    for ing in root.findall('.//ns:ingredient', namespace)
                ],
                'directions': [
                    step.find('ns:direction_description', namespace).text if step.find('ns:direction_description', namespace) is not None else ""
                    for step in root.findall('.//ns:direction', namespace)
                ]
            }
            
            save_recipe_to_db(recipe_data)

        except ET.ParseError:
            print("Failed to parse recipe details XML response.")
    else:
        print(f"Error fetching recipe details for ID {recipe_id}: {response.status_code} - {response.text}")

# Start fetching recipes
fetch_recipes()



Access token obtained successfully.
No recipes found on page 0.


In [9]:
import requests
import time
import xml.etree.ElementTree as ET
import psycopg2
from psycopg2 import sql

# FatSecret API credentials
client_id = '3af760255a2a4932a779f102f2d80b6b'
client_secret = '933419a37c0943018c52066e1feedc20'
auth_url = 'https://oauth.fatsecret.com/connect/token'
api_url = 'https://platform.fatsecret.com/rest/server.api'

# PostgreSQL database credentials
db_config = {
    'dbname': 'recipe_db',
    'user': 'postgres',
    'password': '',  # Replace with your actual password
    'host': 'localhost',
    'port': '5433'
}

# Function to get access token
def get_access_token():
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': 'basic'
    }
    response = requests.post(auth_url, data=data)
    
    if response.status_code == 200:
        access_token = response.json().get('access_token')
        print("Access token obtained successfully.")
        return access_token
    else:
        print("Error getting access token:", response.status_code, response.text)
        return None

# Function to check for duplicate recipes by name
def recipe_exists(recipe_name):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Check if recipe exists by name
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_name,))
        exists = cursor.fetchone() is not None

        cursor.close()
        conn.close()
        return exists

    except Exception as e:
        print("Error checking recipe existence:", e)
        return False

# Function to save recipe data to the PostgreSQL database
def save_recipe_to_db(recipe_data):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()

        # Check if the recipe exists
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_data['name'],))
        result = cursor.fetchone()

        if result:
            # Recipe already exists, get recipe_id
            recipe_id = result[0]
            print(f"Recipe '{recipe_data['name']}' already exists with recipe_id {recipe_id}. Checking ingredients and directions.")
        else:
            # Insert new recipe
            cursor.execute(
                """
                INSERT INTO recipes (name, description, servings, prep_time, cook_time)
                VALUES (%s, %s, %s, %s, %s) RETURNING recipe_id;
                """,
                (recipe_data['name'], recipe_data['description'], recipe_data['servings'], recipe_data['prep_time'], recipe_data['cook_time'])
            )
            recipe_id = cursor.fetchone()[0]
            print(f"Inserted new recipe '{recipe_data['name']}' with recipe_id {recipe_id}.")

            # Insert nutrition facts
            cursor.execute(
                """
                INSERT INTO nutrition_facts (recipe_id, calories, total_fat, protein, carbs)
                VALUES (%s, %s, %s, %s, %s);
                """,
                (recipe_id, recipe_data['calories'], recipe_data['total_fat'], recipe_data['protein'], recipe_data['carbs'])
            )
            print(f"Nutrition facts inserted for recipe_id {recipe_id}")

        # Check and insert ingredients if not present
        cursor.execute("SELECT COUNT(*) FROM ingredients WHERE recipe_id = %s", (recipe_id,))
        ingredients_count = cursor.fetchone()[0]
        if ingredients_count == 0:
            for ingredient_text in recipe_data['ingredients']:
                cursor.execute(
                    """
                    INSERT INTO ingredients (recipe_id, ingredient_text)
                    VALUES (%s, %s);
                    """,
                    (recipe_id, ingredient_text)
                )
            print(f"Inserted ingredients for recipe_id {recipe_id}")
        else:
            print(f"Ingredients already exist for recipe_id {recipe_id}, skipping insertion.")

        # Check and insert directions if not present
        cursor.execute("SELECT COUNT(*) FROM directions WHERE recipe_id = %s", (recipe_id,))
        directions_count = cursor.fetchone()[0]
        if directions_count == 0:
            for step in recipe_data['directions']:
                cursor.execute(
                    """
                    INSERT INTO directions (recipe_id, instruction_text)
                    VALUES (%s, %s);
                    """,
                    (recipe_id, step)
                )
            print(f"Inserted directions for recipe_id {recipe_id}")
        else:
            print(f"Directions already exist for recipe_id {recipe_id}, skipping insertion.")

        # Commit transaction
        conn.commit()
        cursor.close()
        conn.close()
        print(f"Recipe '{recipe_data['name']}' saved to database.")

    except Exception as e:
        print("Error saving recipe to database:", e)

# Function to fetch recipes from FatSecret API
def fetch_recipes():
    access_token = get_access_token()
    
    if not access_token:
        print("Failed to get access token.")
        return
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    
    page_number = 0  
    max_results = 10  
    
    while True:
        params = {
            'method': 'recipes.search',
            'page_number': page_number,
            'max_results': max_results,
            'format': 'xml'
        }
        
        response = requests.get(api_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching page {page_number}: {response.status_code} - {response.text}")
            break

        # Parse XML response with namespace
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            recipes = root.findall('.//ns:recipe', namespace)
            
            if not recipes:
                print(f"No recipes found on page {page_number}.")
                break

            # Process each recipe by getting its full details
            for recipe in recipes:
                recipe_id = recipe.find('ns:recipe_id', namespace).text
                fetch_recipe_details(recipe_id, headers)  # Fetch full details for each recipe

            page_number += 1
            print(f"Fetched page {page_number}.")
            time.sleep(2)

        except ET.ParseError:
            print("Failed to parse XML response.")
            break

# Function to fetch full details for each recipe by ID
def fetch_recipe_details(recipe_id, headers):
    params = {
        'method': 'recipe.get',
        'recipe_id': recipe_id,
        'format': 'xml'
    }
    
    response = requests.get(api_url, headers=headers, params=params)
    
    if response.status_code == 200:
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            
            recipe_data = {
                'name': root.find('.//ns:recipe_name', namespace).text if root.find('.//ns:recipe_name', namespace) is not None else "Unnamed Recipe",
                'description': root.find('.//ns:recipe_description', namespace).text if root.find('.//ns:recipe_description', namespace) is not None else "",
                'servings': root.find('.//ns:number_of_servings', namespace).text if root.find('.//ns:number_of_servings', namespace) is not None else None,
                'prep_time': root.find('.//ns:preparation_time_min', namespace).text if root.find('.//ns:preparation_time_min', namespace) is not None else None,
                'cook_time': root.find('.//ns:cooking_time_min', namespace).text if root.find('.//ns:cooking_time_min', namespace) is not None else None,
                'calories': root.find('.//ns:calories', namespace).text if root.find('.//ns:calories', namespace) is not None else None,
                'total_fat': root.find('.//ns:fat', namespace).text if root.find('.//ns:fat', namespace) is not None else None,
                'protein': root.find('.//ns:protein', namespace).text if root.find('.//ns:protein', namespace) is not None else None,
                'carbs': root.find('.//ns:carbohydrate', namespace).text if root.find('.//ns:carbohydrate', namespace) is not None else None,
                'ingredients': []
            }
            
            # Parse ingredients and construct full ingredient text
            ingredients = root.findall('.//ns:ingredient', namespace)
            for ing in ingredients:
                name = ing.find('ns:food_name', namespace).text if ing.find('ns:food_name', namespace) is not None else ""
                quantity = ing.find('ns:quantity', namespace).text if ing.find('ns:quantity', namespace) is not None else ""
                unit = ing.find('ns:measure', namespace).text if ing.find('ns:measure', namespace) is not None else ""
                
                # Construct the full ingredient text
                full_ingredient_text = f"{quantity} {unit} {name}".strip()
                print(f"Extracted ingredient text: {full_ingredient_text}")
                
                recipe_data['ingredients'].append(full_ingredient_text)

            # Parse directions
            recipe_data['directions'] = [
                step.find('ns:direction_description', namespace).text if step.find('ns:direction_description', namespace) is not None else ""
                for step in root.findall('.//ns:direction', namespace)
            ]
            
            save_recipe_to_db(recipe_data)

        except ET.ParseError:
            print("Failed to parse recipe details XML response.")
    else:
        print(f"Error fetching recipe details for ID {recipe_id}: {response.status_code} - {response.text}")

# Start fetching recipes
fetch_recipes()





Access token obtained successfully.
No recipes found on page 0.


In [ ]:
import requests
import time
import xml.etree.ElementTree as ET
import psycopg2
from psycopg2 import sql

# FatSecret API credentials
client_id = '3af760255a2a4932a779f102f2d80b6b'
client_secret = '933419a37c0943018c52066e1feedc20'
auth_url = 'https://oauth.fatsecret.com/connect/token'
api_url = 'https://platform.fatsecret.com/rest/server.api'

# PostgreSQL database credentials
db_config = {
    'dbname': 'recipe_db',
    'user': 'postgres',
    'password': 'anckul7*K',  # Replace with your actual password
    'host': 'localhost',
    'port': '5433'
}

# Function to get access token
def get_access_token():
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': 'basic'
    }
    response = requests.post(auth_url, data=data)
    
    if response.status_code == 200:
        access_token = response.json().get('access_token')
        print("Access token obtained successfully.")
        return access_token
    else:
        print("Error getting access token:", response.status_code, response.text)
        return None

# Function to check for duplicate recipes by name
def recipe_exists(recipe_name):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Check if recipe exists by name
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_name,))
        exists = cursor.fetchone() is not None

        cursor.close()
        conn.close()
        return exists

    except Exception as e:
        print("Error checking recipe existence:", e)
        return False

# Function to save recipe data to the PostgreSQL database
def save_recipe_to_db(recipe_data):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()

        # Check if the recipe exists
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_data['name'],))
        result = cursor.fetchone()

        if result:
            # Recipe already exists, get recipe_id
            recipe_id = result[0]
            print(f"Recipe '{recipe_data['name']}' already exists with recipe_id {recipe_id}. Checking ingredients and directions.")
        else:
            # Insert new recipe
            cursor.execute(
                """
                INSERT INTO recipes (name, description, servings, prep_time, cook_time)
                VALUES (%s, %s, %s, %s, %s) RETURNING recipe_id;
                """,
                (recipe_data['name'], recipe_data['description'], recipe_data['servings'], recipe_data['prep_time'], recipe_data['cook_time'])
            )
            recipe_id = cursor.fetchone()[0]
            print(f"Inserted new recipe '{recipe_data['name']}' with recipe_id {recipe_id}.")

            # Insert nutrition facts
            cursor.execute(
                """
                INSERT INTO nutrition_facts (recipe_id, calories, total_fat, protein, carbs)
                VALUES (%s, %s, %s, %s, %s);
                """,
                (recipe_id, recipe_data['calories'], recipe_data['total_fat'], recipe_data['protein'], recipe_data['carbs'])
            )
            print(f"Nutrition facts inserted for recipe_id {recipe_id}")

        # Insert or update ingredients
        for ingredient in recipe_data['ingredients']:
            cursor.execute(
                """
                INSERT INTO ingredients (recipe_id, ingredient_text)
                VALUES (%s, %s)
                ON CONFLICT (recipe_id, ingredient_text) DO NOTHING;
                """,
                (recipe_id, ingredient['name'])
            )
        print(f"Inserted or updated ingredients for recipe_id {recipe_id}")

        # Insert or update directions
        cursor.execute("DELETE FROM directions WHERE recipe_id = %s", (recipe_id,))
        for step in recipe_data['directions']:
            cursor.execute(
                """
                INSERT INTO directions (recipe_id, instruction_text)
                VALUES (%s, %s);
                """,
                (recipe_id, step)
            )
        print(f"Inserted directions for recipe_id {recipe_id}")

        # Commit transaction
        conn.commit()
        cursor.close()
        conn.close()
        print(f"Recipe '{recipe_data['name']}' saved to database.")

    except Exception as e:
        print("Error saving recipe to database:", e)

# Function to fetch recipes from FatSecret API
def fetch_recipes():
    access_token = get_access_token()
    
    if not access_token:
        print("Failed to get access token.")
        return
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    
    page_number = 0  
    max_results = 10  
    
    while True:
        params = {
            'method': 'recipes.search',
            'page_number': page_number,
            'max_results': max_results,
            'format': 'xml'
        }
        
        response = requests.get(api_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching page {page_number}: {response.status_code} - {response.text}")
            break

        # Parse XML response with namespace
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            recipes = root.findall('.//ns:recipe', namespace)
            
            if not recipes:
                print(f"No recipes found on page {page_number}.")
                break

            # Process each recipe by getting its full details
            for recipe in recipes:
                recipe_id = recipe.find('ns:recipe_id', namespace).text
                fetch_recipe_details(recipe_id, headers)  # Fetch full details for each recipe

            page_number += 1
            print(f"Fetched page {page_number}.")
            time.sleep(2)

        except ET.ParseError:
            print("Failed to parse XML response.")
            break

# Function to fetch full details for each recipe by ID
def fetch_recipe_details(recipe_id, headers):
    params = {
        'method': 'recipe.get',
        'recipe_id': recipe_id,
        'format': 'xml'
    }
    
    response = requests.get(api_url, headers=headers, params=params)
    
    if response.status_code == 200:
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            
            recipe_data = {
                'name': root.find('.//ns:recipe_name', namespace).text if root.find('.//ns:recipe_name', namespace) is not None else "Unnamed Recipe",
                'description': root.find('.//ns:recipe_description', namespace).text if root.find('.//ns:recipe_description', namespace) is not None else "",
                'servings': int(root.find('.//ns:number_of_servings', namespace).text) if root.find('.//ns:number_of_servings', namespace) is not None else None,
                'prep_time': int(root.find('.//ns:preparation_time_min', namespace).text) if root.find('.//ns:preparation_time_min', namespace) is not None else None,
                'cook_time': int(root.find('.//ns:cooking_time_min', namespace).text) if root.find('.//ns:cooking_time_min', namespace) is not None else None,
                'calories': int(root.find('.//ns:calories', namespace).text) if root.find('.//ns:calories', namespace) is not None else None,
                'total_fat': float(root.find('.//ns:fat', namespace).text) if root.find('.//ns:fat', namespace) is not None else None,
                'protein': float(root.find('.//ns:protein', namespace).text) if root.find('.//ns:protein', namespace) is not None else None,
                'carbs': float(root.find('.//ns:carbohydrate', namespace).text) if root.find('.//ns:carbohydrate', namespace) is not None else None,
                'ingredients': [
                    {
                        'name': ing.find('ns:food_name', namespace).text if ing.find('ns:food_name', namespace) is not None else "Unknown Ingredient"
                    }
                    for ing in root.findall('.//ns:ingredient', namespace)
                ],
                'directions': [
                    step.find('ns:direction_description', namespace).text if step.find('ns:direction_description', namespace) is not None else ""
                    for step in root.findall('.//ns:direction', namespace)
                ]
            }
            
            save_recipe_to_db(recipe_data)

        except ET.ParseError:
            print("Failed to parse recipe details XML response.")
    else:
        print(f"Error fetching recipe details for ID {recipe_id}: {response.status_code} - {response.text}")

# Start fetching recipes
fetch_recipes()



Access token obtained successfully.
No recipes found on page 0.


In [ ]:
import requests
import time
import xml.etree.ElementTree as ET
import psycopg2
from psycopg2 import sql
import csv
import os

# FatSecret API credentials
client_id = '3af760255a2a4932a779f102f2d80b6b'
client_secret = '933419a37c0943018c52066e1feedc20'
auth_url = 'https://oauth.fatsecret.com/connect/token'
api_url = 'https://platform.fatsecret.com/rest/server.api'

# PostgreSQL database credentials
db_config = {
    'dbname': 'recipe_db',
    'user': 'postgres',
    'password': 'anckul7*K',  # Replace with your actual password
    'host': 'localhost',
    'port': '5433'
}

seen_recipes = set()

# Function to get access token
def get_access_token():
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': 'basic'
    }
    response = requests.post(auth_url, data=data)
    
    if response.status_code == 200:
        access_token = response.json().get('access_token')
        print("Access token obtained successfully.")
        return access_token
    else:
        print("Error getting access token:", response.status_code, response.text)
        return None

# Function to check for duplicate recipes by name
def recipe_exists(recipe_name):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Check if recipe exists by name
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_name,))
        exists = cursor.fetchone() is not None

        cursor.close()
        conn.close()
        return exists

    except Exception as e:
        print("Error checking recipe existence:", e)
        return False

# Function to save recipe data to the PostgreSQL database
def save_recipe_to_db(recipe_data):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()

        # Check if the recipe exists
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_data['name'],))
        result = cursor.fetchone()

        if result:
            # Recipe already exists, get recipe_id
            recipe_id = result[0]
            print(f"Recipe '{recipe_data['name']}' already exists with recipe_id {recipe_id}. Checking ingredients and directions.")
        else:
            # Insert new recipe
            cursor.execute(
                """
                INSERT INTO recipes (name, description, servings, prep_time, cook_time)
                VALUES (%s, %s, %s, %s, %s) RETURNING recipe_id;
                """,
                (recipe_data['name'], recipe_data['description'], recipe_data['servings'], recipe_data['prep_time'], recipe_data['cook_time'])
            )
            recipe_id = cursor.fetchone()[0]
            print(f"Inserted new recipe '{recipe_data['name']}' with recipe_id {recipe_id}.")

            # Insert nutrition facts
            cursor.execute(
                """
                INSERT INTO nutrition_facts (recipe_id, calories, total_fat, protein, carbs)
                VALUES (%s, %s, %s, %s, %s);
                """,
                (recipe_id, recipe_data['calories'], recipe_data['total_fat'], recipe_data['protein'], recipe_data['carbs'])
            )
            print(f"Nutrition facts inserted for recipe_id {recipe_id}")

        # Insert or update ingredients
        for ingredient in recipe_data['ingredients']:
            cursor.execute(
                """
                INSERT INTO ingredients (recipe_id, ingredient_text)
                VALUES (%s, %s)
                ON CONFLICT (recipe_id, ingredient_text) DO NOTHING;
                """,
                (recipe_id, ingredient['name'])
            )
        print(f"Inserted or updated ingredients for recipe_id {recipe_id}")

        # Insert directions (delete old ones and insert new ones)
        cursor.execute("DELETE FROM directions WHERE recipe_id = %s", (recipe_id,))
        for step in recipe_data['directions']:
            cursor.execute(
                """
                INSERT INTO directions (recipe_id, instruction_text)
                VALUES (%s, %s);
                """,
                (recipe_id, step)
            )
        print(f"Inserted directions for recipe_id {recipe_id}")

        # Commit transaction
        conn.commit()
        cursor.close()
        conn.close()
        print(f"Recipe '{recipe_data['name']}' saved to database.")

    except Exception as e:
        print("Error saving recipe to database:", e)

# Function to save recipes to CSV and avoid duplicates
def save_recipes_to_csv(recipes):
    file_exists = os.path.isfile(output_file)
    with open(output_file, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(["Recipe ID", "Recipe Name", "Description", "Preparation Time", "Cooking Time"])  # Header
        for recipe in recipes:
            recipe_id = recipe['recipe_id']
            if recipe_id not in seen_recipes:  # Check if it's a new recipe
                seen_recipes.add(recipe_id)
                writer.writerow([
                    recipe_id,
                    recipe.get('recipe_name', ''),
                    recipe.get('recipe_description', ''),
                    recipe.get('preparation_time', ''),
                    recipe.get('cooking_time', '')
                ])

# Function to fetch recipes from FatSecret API
def fetch_recipes():
    access_token = get_access_token()
    
    if not access_token:
        print("Failed to get access token.")
        return
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    
    page_number = 0  
    max_results = 10  
    total_fetched = 0
    
    while True:
        params = {
            'method': 'recipes.search',
            'page_number': page_number,
            'max_results': max_results,
            'format': 'xml'
        }
        
        response = requests.get(api_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching page {page_number}: {response.status_code} - {response.text}")
            break

        # Parse XML response with namespace
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            recipes = root.findall('.//ns:recipe', namespace)
            
            if not recipes:
                print(f"No recipes found on page {page_number}.")
                break

            # Extract recipes data
            recipes_data = []
            for recipe in recipes:
                recipe_id = recipe.find('ns:recipe_id', namespace).text
                recipe_name = recipe.find('ns:recipe_name', namespace).text
                recipe_description = recipe.find('ns:recipe_description', namespace).text
                preparation_time = recipe.find('ns:preparation_time_min', namespace).text if recipe.find('ns:preparation_time_min', namespace) is not None else ''
                cooking_time = recipe.find('ns:cooking_time_min', namespace).text if recipe.find('ns:cooking_time_min', namespace) is not None else ''

                # Append recipe data to the list
                recipes_data.append({
                    'recipe_id': recipe_id,
                    'recipe_name': recipe_name,
                    'recipe_description': recipe_description,
                    'preparation_time': preparation_time,
                    'cooking_time': cooking_time
                })

            # Save to CSV if recipes are found
            save_recipes_to_csv(recipes_data)
            total_fetched += len(recipes_data)

            # Process each recipe by getting its full details
            for recipe in recipes:
                recipe_id = recipe.find('ns:recipe_id', namespace).text
                fetch_recipe_details(recipe_id, headers)  # Fetch full details for each recipe

            page_number += 1
            print(f"Fetched page {page_number}, total recipes so far: {total_fetched}")
            time.sleep(2)

        except ET.ParseError:
            print("Failed to parse XML response.")
            break

# Function to fetch full details for each recipe by ID
def fetch_recipe_details(recipe_id, headers):
    params = {
        'method': 'recipe.get',
        'recipe_id': recipe_id,
        'format': 'xml'
    }
    
    response = requests.get(api_url, headers=headers, params=params)
    
    if response.status_code == 200:
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            
            recipe_data = {
                'name': root.find('.//ns:recipe_name', namespace).text if root.find('.//ns:recipe_name', namespace) is not None else "Unnamed Recipe",
                'description': root.find('.//ns:recipe_description', namespace).text if root.find('.//ns:recipe_description', namespace) is not None else "",
                'servings': float(root.find('.//ns:number_of_servings', namespace).text) if root.find('.//ns:number_of_servings', namespace) is not None else None,
                'prep_time': float(root.find('.//ns:preparation_time_min', namespace).text) if root.find('.//ns:preparation_time_min', namespace) is not None else None,
                'cook_time': float(root.find('.//ns:cooking_time_min', namespace).text) if root.find('.//ns:cooking_time_min', namespace) is not None else None,
                'calories': float(root.find('.//ns:calories', namespace).text) if root.find('.//ns:calories', namespace) is not None else None,
                'total_fat': float(root.find('.//ns:fat', namespace).text) if root.find('.//ns:fat', namespace) is not None else None,
                'protein': float(root.find('.//ns:protein', namespace).text) if root.find('.//ns:protein', namespace) is not None else None,
                'carbs': float(root.find('.//ns:carbohydrate', namespace).text) if root.find('.//ns:carbohydrate', namespace) is not None else None,
                'ingredients': [
                    {
                        'name': ing.find('ns:food_name', namespace).text if ing.find('ns:food_name', namespace) is not None else "Unknown Ingredient"
                    }
                    for ing in root.findall('.//ns:ingredient', namespace)
                ],
                'directions': [
                    step.find('ns:direction_description', namespace).text if step.find('ns:direction_description', namespace) is not None else ""
                    for step in root.findall('.//ns:direction', namespace)
                ]
            }

            
            save_recipe_to_db(recipe_data)

        except ET.ParseError:
            print("Failed to parse recipe details XML response.")
    else:
        print(f"Error fetching recipe details for ID {recipe_id}: {response.status_code} - {response.text}")

# Start fetching recipes
fetch_recipes()

print(f"Total unique recipes saved: {len(seen_recipes)}")


Access token obtained successfully.
Recipe 'Peanut Butter Banana Protein Smoothie' already exists with recipe_id 45. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 45
Inserted directions for recipe_id 45
Recipe 'Peanut Butter Banana Protein Smoothie' saved to database.
Recipe 'Chocolate Peanut Butter Squares' already exists with recipe_id 46. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 46
Inserted directions for recipe_id 46
Recipe 'Chocolate Peanut Butter Squares' saved to database.
Recipe 'Kale Salad' already exists with recipe_id 47. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 47
Inserted directions for recipe_id 47
Recipe 'Kale Salad' saved to database.
Recipe 'Protein Balls' already exists with recipe_id 48. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 48
Inserted directions for recipe_id 48
Recipe 'Protein Balls' saved to database

In [9]:
import requests
import time
import xml.etree.ElementTree as ET
import psycopg2
from psycopg2 import sql

# FatSecret API credentials
client_id = '3af760255a2a4932a779f102f2d80b6b'
client_secret = '933419a37c0943018c52066e1feedc20'
auth_url = 'https://oauth.fatsecret.com/connect/token'
api_url = 'https://platform.fatsecret.com/rest/server.api'

# PostgreSQL database credentials
db_config = {
    'dbname': 'recipe_db',
    'user': 'postgres',
    'password': 'anckul7*K',  # Replace with your actual password
    'host': 'localhost',
    'port': '5433'
}

# Function to get access token
def get_access_token():
    data = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'scope': 'basic'
    }
    response = requests.post(auth_url, data=data)
    
    if response.status_code == 200:
        access_token = response.json().get('access_token')
        print("Access token obtained successfully.")
        return access_token
    else:
        print("Error getting access token:", response.status_code, response.text)
        return None

# Function to check for duplicate recipes by name
def recipe_exists(recipe_name):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Check if recipe exists by name
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_name,))
        exists = cursor.fetchone() is not None

        cursor.close()
        conn.close()
        return exists

    except Exception as e:
        print("Error checking recipe existence:", e)
        return False

# Function to save recipe data to the PostgreSQL database
def save_recipe_to_db(recipe_data):
    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()

        # Check if the recipe exists
        cursor.execute("SELECT recipe_id FROM recipes WHERE name = %s", (recipe_data['name'],))
        result = cursor.fetchone()

        if result:
            # Recipe already exists, get recipe_id
            recipe_id = result[0]
            print(f"Recipe '{recipe_data['name']}' already exists with recipe_id {recipe_id}. Checking ingredients and directions.")
        else:
            # Insert new recipe
            cursor.execute(
                """
                INSERT INTO recipes (name, description, servings, prep_time, cook_time)
                VALUES (%s, %s, %s, %s, %s) RETURNING recipe_id;
                """,
                (recipe_data['name'], recipe_data['description'], recipe_data['servings'], recipe_data['prep_time'], recipe_data['cook_time'])
            )
            recipe_id = cursor.fetchone()[0]
            print(f"Inserted new recipe '{recipe_data['name']}' with recipe_id {recipe_id}.")

            # Insert nutrition facts
            cursor.execute(
                """
                INSERT INTO nutrition_facts (recipe_id, calories, total_fat, protein, carbs)
                VALUES (%s, %s, %s, %s, %s);
                """,
                (recipe_id, recipe_data['calories'], recipe_data['total_fat'], recipe_data['protein'], recipe_data['carbs'])
            )
            print(f"Nutrition facts inserted for recipe_id {recipe_id}")

        # Insert or update ingredients
        for ingredient in recipe_data['ingredients']:
            cursor.execute(
                """
                INSERT INTO ingredients (recipe_id, ingredient_text)
                VALUES (%s, %s)
                ON CONFLICT (recipe_id, ingredient_text) DO NOTHING;
                """,
                (recipe_id, ingredient['name'])
            )
        print(f"Inserted or updated ingredients for recipe_id {recipe_id}")

        # Insert directions (delete old ones and insert new ones)
        cursor.execute("DELETE FROM directions WHERE recipe_id = %s", (recipe_id,))
        for step in recipe_data['directions']:
            cursor.execute(
                """
                INSERT INTO directions (recipe_id, instruction_text)
                VALUES (%s, %s);
                """,
                (recipe_id, step)
            )
        print(f"Inserted directions for recipe_id {recipe_id}")

        # Commit transaction
        conn.commit()
        cursor.close()
        conn.close()
        print(f"Recipe '{recipe_data['name']}' saved to database.")

    except Exception as e:
        print("Error saving recipe to database:", e)

# Function to fetch recipes from FatSecret API
def fetch_recipes():
    access_token = get_access_token()
    
    if not access_token:
        print("Failed to get access token.")
        return
    
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }
    
    page_number = 0  
    max_results = 10  
    total_fetched = 0
    
    while True:
        params = {
            'method': 'recipes.search',
            'page_number': page_number,
            'max_results': max_results,
            'format': 'xml'
        }
        
        response = requests.get(api_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Error fetching page {page_number}: {response.status_code} - {response.text}")
            break

        # Parse XML response with namespace
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            recipes = root.findall('.//ns:recipe', namespace)
            
            if not recipes:
                print(f"No recipes found on page {page_number}.")
                break

            # Extract recipes data
            recipes_data = []
            for recipe in recipes:
                recipe_id = recipe.find('ns:recipe_id', namespace).text
                recipe_name = recipe.find('ns:recipe_name', namespace).text
                recipe_description = recipe.find('ns:recipe_description', namespace).text
                preparation_time = recipe.find('ns:preparation_time_min', namespace).text if recipe.find('ns:preparation_time_min', namespace) is not None else ''
                cooking_time = recipe.find('ns:cooking_time_min', namespace).text if recipe.find('ns:cooking_time_min', namespace) is not None else ''

                # Append recipe data to the list
                recipes_data.append({
                    'recipe_id': recipe_id,
                    'recipe_name': recipe_name,
                    'recipe_description': recipe_description,
                    'preparation_time': preparation_time,
                    'cooking_time': cooking_time
                })

            total_fetched += len(recipes_data)

            # Process each recipe by getting its full details
            for recipe in recipes:
                recipe_id = recipe.find('ns:recipe_id', namespace).text
                fetch_recipe_details(recipe_id, headers)  # Fetch full details for each recipe

            page_number += 1
            print(f"Fetched page {page_number}, total recipes so far: {total_fetched}")
            time.sleep(2)

        except ET.ParseError:
            print("Failed to parse XML response.")
            break

# Function to fetch full details for each recipe by ID
def fetch_recipe_details(recipe_id, headers):
    params = {
        'method': 'recipe.get',
        'recipe_id': recipe_id,
        'format': 'xml'
    }
    
    response = requests.get(api_url, headers=headers, params=params)
    
    if response.status_code == 200:
        try:
            root = ET.fromstring(response.text)
            namespace = {'ns': 'http://platform.fatsecret.com/api/1.0/'}
            
            recipe_data = {
                'name': root.find('.//ns:recipe_name', namespace).text if root.find('.//ns:recipe_name', namespace) is not None else "Unnamed Recipe",
                'description': root.find('.//ns:recipe_description', namespace).text if root.find('.//ns:recipe_description', namespace) is not None else "",
                'servings': float(root.find('.//ns:number_of_servings', namespace).text) if root.find('.//ns:number_of_servings', namespace) is not None else None,
                'prep_time': float(root.find('.//ns:preparation_time_min', namespace).text) if root.find('.//ns:preparation_time_min', namespace) is not None else None,
                'cook_time': float(root.find('.//ns:cooking_time_min', namespace).text) if root.find('.//ns:cooking_time_min', namespace) is not None else None,
                'calories': float(root.find('.//ns:calories', namespace).text) if root.find('.//ns:calories', namespace) is not None else None,
                'total_fat': float(root.find('.//ns:fat', namespace).text) if root.find('.//ns:fat', namespace) is not None else None,
                'protein': float(root.find('.//ns:protein', namespace).text) if root.find('.//ns:protein', namespace) is not None else None,
                'carbs': float(root.find('.//ns:carbohydrate', namespace).text) if root.find('.//ns:carbohydrate', namespace) is not None else None,
                'ingredients': [
                    {
                        'name': ing.find('ns:food_name', namespace).text if ing.find('ns:food_name', namespace) is not None else "Unknown Ingredient"
                    }
                    for ing in root.findall('.//ns:ingredient', namespace)
                ],
                'directions': [
                    step.find('ns:direction_description', namespace).text if step.find('ns:direction_description', namespace) is not None else ""
                    for step in root.findall('.//ns:direction', namespace)
                ]
            }

            save_recipe_to_db(recipe_data)

        except ET.ParseError:
            print("Failed to parse recipe details XML response.")
    else:
        print(f"Error fetching recipe details for ID {recipe_id}: {response.status_code} - {response.text}")

# Start fetching recipes
fetch_recipes()


Access token obtained successfully.
Recipe 'Peanut Butter Banana Protein Smoothie' already exists with recipe_id 45. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 45
Inserted directions for recipe_id 45
Recipe 'Peanut Butter Banana Protein Smoothie' saved to database.
Recipe 'Chocolate Peanut Butter Squares' already exists with recipe_id 46. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 46
Inserted directions for recipe_id 46
Recipe 'Chocolate Peanut Butter Squares' saved to database.
Recipe 'Kale Salad' already exists with recipe_id 47. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 47
Inserted directions for recipe_id 47
Recipe 'Kale Salad' saved to database.
Recipe 'Protein Balls' already exists with recipe_id 48. Checking ingredients and directions.
Inserted or updated ingredients for recipe_id 48
Inserted directions for recipe_id 48
Recipe 'Protein Balls' saved to database

In [11]:
import psycopg2
import pandas as pd
from psycopg2 import sql

# PostgreSQL database credentials
db_config = {
    'dbname': 'recipe_db',
    'user': 'postgres',
    'password': 'anckul7*K',  # Replace with your actual password
    'host': 'localhost',
    'port': '5433'
}

# Connect to PostgreSQL
try:
    conn = psycopg2.connect(**db_config)
    print("Database connection successful.")
except Exception as e:
    print("Error connecting to database:", e)
    exit()

# Function to read data from a table and remove duplicates
def read_table_data(connection, table_name):
    query = f"SELECT * FROM {table_name};"
    df = pd.read_sql_query(query, connection)
    df = df.drop_duplicates()  # Remove duplicate rows
    return df

# Read data from tables
try:
    recipes_df = read_table_data(conn, 'recipes')
    ingredients_df = read_table_data(conn, 'ingredients')
    nutrition_facts_df = read_table_data(conn, 'nutrition_facts')
    directions_df = read_table_data(conn, 'directions')

    # Create an Excel file with each table in a separate sheet
    with pd.ExcelWriter('recipe_database.xlsx') as writer:
        recipes_df.to_excel(writer, sheet_name='Recipes', index=False)
        ingredients_df.to_excel(writer, sheet_name='Ingredients', index=False)
        nutrition_facts_df.to_excel(writer, sheet_name='Nutrition_Facts', index=False)
        directions_df.to_excel(writer, sheet_name='Directions', index=False)

    print("Excel file 'recipe_database.xlsx' created successfully with all tables.")

except Exception as e:
    print("Error while fetching data or writing to Excel:", e)

# Close the database connection
finally:
    conn.close()
    print("Database connection closed.")



Database connection successful.


C:\Users\ancha\AppData\Local\Temp\ipykernel_29768\2952382816.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, connection)


Excel file 'recipe_database.xlsx' created successfully with all tables.
Database connection closed.
